In [16]:
!pip install openmeteo_requests

import openmeteo_requests
from datetime import datetime

class IncreaseSpeed:
    def __init__(self, current_speed: int, max_speed: int, step=10):
        self.current_speed = current_speed
        self.max_speed = max_speed
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.current_speed >= self.max_speed:
            raise StopIteration

        new_speed = self.current_speed + self.step

        if new_speed > self.max_speed:
            new_speed = self.max_speed

        self.current_speed = new_speed
        return self.current_speed


class DecreaseSpeed:
    def __init__(self, current_speed: int, min_speed: int, step=10):
        self.current_speed = current_speed
        self.min_speed = min_speed
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.current_speed <= self.min_speed:
            raise StopIteration

        new_speed = self.current_speed - self.step

        if new_speed < self.min_speed:
            new_speed = self.min_speed

        self.current_speed = new_speed
        return self.current_speed


class Car:
    total_cars_on_road = 0

    def __init__(self, max_speed: int, current_speed=0):
        self.max_speed = max_speed
        self.current_speed = current_speed
        self.state = "on_road" if current_speed > 0 else "parking"

        if self.state == "on_road":
            Car.total_cars_on_road += 1

    def accelerate(self, upper_border=None, step=10):
        if self.state != "on_road":
            self.state = "on_road"
            Car.total_cars_on_road += 1

        target_speed = upper_border if upper_border is not None else self.max_speed

        if target_speed > self.max_speed:
            target_speed = self.max_speed
        elif target_speed < self.current_speed:
            print(f"INFO: Target speed {target_speed} is lower than current speed {self.current_speed}")
            return f"Current speed: {self.current_speed} km/h"

        initial_speed = self.current_speed

        if upper_border is not None and upper_border > self.current_speed:
            speed_iterator = IncreaseSpeed(self.current_speed, target_speed, step)

            for new_speed in speed_iterator:
                self.current_speed = new_speed
                print(f"INFO: Speed increases by {step}")

            print(f"INFO: The speed of this car has been increased from {initial_speed} to {self.current_speed}")
        else:
            new_speed = min(self.current_speed + step, target_speed)
            self.current_speed = new_speed
            print(f"INFO: Speed increases by {step}")
            print(f"INFO: The speed of this car has been increased from {initial_speed} to {self.current_speed}")

        return f"Current speed: {self.current_speed} km/h"

    def brake(self, lower_border=None, step=10):
        # Check if car is on road
        if self.state != "on_road":
            print("INFO: Car is in parking, cannot brake")
            return f"Current speed: {self.current_speed} km/h"

        target_speed = lower_border if lower_border is not None else 0

        if target_speed < 0:
            target_speed = 0
        elif target_speed > self.current_speed:
            print(f"INFO: Target speed {target_speed} is higher than current speed {self.current_speed}")
            return f"Current speed: {self.current_speed} km/h"

        initial_speed = self.current_speed

        if lower_border is not None and lower_border < self.current_speed:
            speed_iterator = DecreaseSpeed(self.current_speed, target_speed, step)

            for new_speed in speed_iterator:
                self.current_speed = new_speed
                print(f"INFO: Speed decreases by {step}")

            print(f"INFO: The speed of this car has been decreased from {initial_speed} to {self.current_speed}")
        else:
            new_speed = max(self.current_speed - step, target_speed)
            self.current_speed = new_speed
            print(f"INFO: Speed decreases by {step}")
            print(f"INFO: The speed of this car has been decreased from {initial_speed} to {self.current_speed}")

        return f"Current speed: {self.current_speed} km/h"

    def parking(self):
        if self.state == "parking":
            print("INFO: Car is already in parking")
            return

        if self.current_speed > 0:
            self.brake(0)

        self.state = "parking"
        Car.total_cars_on_road -= 1
        print("Parking the car...")

        return f"Car is now in parking. Current speed: {self.current_speed} km/h"

    @classmethod
    def total_cars(cls):
        return cls.total_cars_on_road

    @staticmethod
    def show_weather():
        openmeteo = openmeteo_requests.Client()
        url = "https://api.open-meteo.com/v1/forecast"
        params = {
            "latitude": 59.9386,  # for St.Petersburg
            "longitude": 30.3141,  # for St.Petersburg
            "current": ["temperature_2m", "apparent_temperature", "rain", "wind_speed_10m"],
            "wind_speed_unit": "ms",
            "timezone": "Europe/Moscow"
        }

        response = openmeteo.weather_api(url, params=params)[0]

        current = response.Current()
        current_temperature_2m = current.Variables(0).Value()
        current_apparent_temperature = current.Variables(1).Value()
        current_rain = current.Variables(2).Value()
        current_wind_speed_10m = current.Variables(3).Value()

        print(f"Current time: {datetime.fromtimestamp(current.Time()+response.UtcOffsetSeconds())} {response.TimezoneAbbreviation().decode()}")
        print(f"Current temperature: {round(current_temperature_2m, 0)} C")
        print(f"Current apparent_temperature: {round(current_apparent_temperature, 0)} C")
        print(f"Current rain: {current_rain} mm")
        print(f"Current wind_speed: {round(current_wind_speed_10m, 1)} m/s")

In [20]:
car1 = Car(100, 20) # max_speed = 100, initial speed = 5
car2 = Car(60, 30) # max_speed = 60, initial speed = 30
car3 = Car(100, 0) # a car that is off road upon creation
print(f"Total cars on road: {Car.total_cars()}")

Total cars on road: 8


In [21]:
car1.accelerate(100)

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 20 to 100


'Current speed: 100 km/h'

In [22]:
car2.accelerate(50)

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 30 to 50


'Current speed: 50 km/h'

In [25]:
print("Speed of car 1:", car1.current_speed)

Speed of car 1: 100


In [26]:
print("Speed of car 2:", car2.current_speed)

Speed of car 2: 50


In [31]:
car1.brake(10)

INFO: Speed decreases by 10
INFO: The speed of this car has been decreased from 10 to 10


'Current speed: 10 km/h'

In [36]:
car2.brake(0)
print("Total cars on road:", Car.total_cars())
car2.parking()
print("Total cars on road:", Car.total_cars())

INFO: Car is in parking, cannot brake
Total cars on road: 7
INFO: Car is already in parking
Total cars on road: 7


In [39]:
car3.accelerate(80)# car3 is now on the road
car3.show_weather()
print("Total cars on road:", Car.total_cars())

INFO: Speed increases by 10
INFO: The speed of this car has been increased from 80 to 80
Current time: 2026-03-26 00:00:00 GMT+3
Current temperature: 10.0 C
Current apparent_temperature: 5.0 C
Current rain: 0.0 mm
Current wind_speed: 5.6 m/s
Total cars on road: 8


In [41]:
car2.accelerate(10) # # car2 goes from parking on the road
print("Total cars on road:", Car.total_cars())

INFO: Speed increases by 10
INFO: The speed of this car has been increased from 10 to 10
Total cars on road: 9


In [43]:
Car.show_weather()

Current time: 2026-03-26 00:00:00 GMT+3
Current temperature: 10.0 C
Current apparent_temperature: 5.0 C
Current rain: 0.0 mm
Current wind_speed: 5.6 m/s
